In [1]:
!pip install -q transformers accelerate datasets peft

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [4]:
from google.colab import files
uploaded = files.upload()

In [5]:
file_name = "revisedsupportassistant_data.txt"  

with open(file_name, "r", encoding="utf-8") as f:
    data = f.read()

In [6]:
from datasets import Dataset

conversations = data.strip().split("\n\n")  

dataset = Dataset.from_dict({"text": conversations})

In [7]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully ✅")

In [8]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

In [9]:
!pip install -q peft==0.8.2

In [10]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [11]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",              
    per_device_train_batch_size=1,       
    gradient_accumulation_steps=4,       
    num_train_epochs=3,                  
    logging_steps=10,                    
    save_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

RAG IMPLEMENTATION

In [1]:
!pip install langchain chromadb sentence-transformers pypdf

In [2]:
!pip install -q langchain-community
!pip install -q langchain-text-splitters

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [4]:
from google.colab import files
uploaded = files.upload()

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Merged_Ecommerce_Guide.pdf")
documents = loader.load()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

In [7]:
!pip install -q langchain-community sentence-transformers

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

In [10]:
!pip install -U langchain langchain-community

In [11]:
from langchain_community.vectorstores import Chroma

In [12]:
db = Chroma.from_documents(docs, embeddings)

In [13]:
retriever = db.as_retriever()

In [14]:
def rag_response(query):
    docs = retriever.invoke(query)

    
    filtered_docs = []
    for doc in docs:
        if "refund" in doc.page_content.lower():
            filtered_docs.append(doc)

    
    if len(filtered_docs) == 0:
        filtered_docs = docs

    context = "\n\n".join([doc.page_content for doc in filtered_docs])

    prompt = f"""You are a professional e-commerce support assistant.

Answer ONLY using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.3,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in response:
        response = response.split("Answer:")[-1]

    if "Question:" in response:
        response = response.split("Question:")[0]

    return response.strip()

In [15]:
retriever = db.as_retriever(search_kwargs={"k": 4})

In [41]:
def ft_response(query):
    prompt = f"<|user|>\n{query}\n<|assistant|>"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def rag_response(query):
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])

    
    prompt = f"""<|user|>
You are a helpful E-commerce Support Agent. Using your knowledge (provided below), answer the customer directly.

STRICT RULES:
1. Do NOT mention words like "the context", "according to the document", or "based on the text".
2. Start your reply directly (e.g., "Hello! I can help you with that..." or "Yes, you can...").
3. Act as if you are a real person speaking to the customer.

Knowledge:
{context}

Question: {query}
<|assistant|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.2, 
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    
    if "Question:" in response:
        response = response.split("Question:")[0]

    return response.strip()




In [42]:
questions = [
    "I live in a remote area. How long does it usually take for delivery?",
    "Can I exchange a product instead of returning it?",
    "I lost the original packaging. Can I still return the product?",
    "I want to merge two accounts. Is that possible",
    "I want to use my loyalty points for a ₹900 order, but it's not working. Why?"

]

In [43]:
compare_models(questions)